In [1]:
import pandas as pd
import os
import time
import warnings
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.simbad import Simbad

# 1. Configure SIMBAD
custom_simbad = Simbad()
custom_simbad.add_votable_fields('otype') 

# 2. Load data, forcing source_id to string to prevent floating-point destruction
candidates_path = '../results/candidates/multimodal_candidates.csv'
clean_data_path = '../data/intermediate/gaia_clean.csv'

df_candidates = pd.read_csv(candidates_path, dtype={'source_id': str})
df_clean = pd.read_csv(clean_data_path, dtype={'source_id': str})

# 3. Merge coordinates
top_50 = df_candidates.head(50).copy()
top_50 = pd.merge(top_50, df_clean[['source_id', 'ra', 'dec']], on='source_id', how='left')
top_50 = top_50.dropna(subset=['ra', 'dec']) # Drop merge failures

print(f"Querying SIMBAD for {len(top_50)} candidates...")
results = []

for index, row in top_50.iterrows():
    coord = SkyCoord(ra=row['ra']*u.degree, dec=row['dec']*u.degree, frame='icrs')
    
    try:
        # Catch and hide the spammy "No Results" warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result_table = custom_simbad.query_region(coord, radius=2 * u.arcsec)
        
        if result_table is not None and len(result_table) > 0:
            # Safely grab columns, converting to uppercase for safe matching
            cols = [c.upper() for c in result_table.colnames]
            
            main_id = result_table[result_table.colnames[cols.index('MAIN_ID')]][0] if 'MAIN_ID' in cols else "Unknown_ID"
            otype = result_table[result_table.colnames[cols.index('OTYPE')]][0] if 'OTYPE' in cols else "Unknown_Type"
            
            # Decode bytes if the server returned raw byte strings
            main_id = main_id.decode('utf-8') if isinstance(main_id, bytes) else str(main_id)
            otype = otype.decode('utf-8') if isinstance(otype, bytes) else str(otype)
            
            results.append({
                'source_id': row['source_id'],
                'simbad_id': main_id,
                'object_type': otype,
                'known_object': True,
                'match_status': 'SIMBAD positional match'
            })
        else:
            # This is a genuine, unclassified anomaly!
            results.append({
                'source_id': row['source_id'],
                'simbad_id': 'None',
                'object_type': 'No SIMBAD match',
                'known_object': False,
                'match_status': 'No match within 2 arcsec'
            })
            
    except Exception as e:
        print(f"Error querying object {row['source_id']}: {e}")
        results.append({
            'source_id': row['source_id'],
            'simbad_id': 'Error',
            'object_type': 'Error',
            'known_object': False,
            'match_status': 'Query error'
        })
        
    time.sleep(0.5) # Be polite to SIMBAD servers

# 4. Format and display
df_simbad = pd.DataFrame(results)
final_candidates = pd.merge(top_50, df_simbad, on='source_id', how='left')

output_path = '../results/candidates/validated_candidates.csv'
final_candidates.to_csv(output_path, index=False)

print(f"\nDiscovered {final_candidates['known_object'].sum()} previously classified objects.")
display(final_candidates[['source_id', 'score_All_Fused', 'simbad_id', 'object_type']].head(15))

<project-directory>/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Querying SIMBAD for 50 candidates...

Discovered 5 previously classified objects.


,source_id,score_All_Fused,simbad_id,object_type
0,138851516677358336,1.000000,None,No SIMBAD match
1,138837188665895296,0.978328,None,No SIMBAD match
2,138844511586077312,0.969342,None,No SIMBAD match
3,138943978733072768,0.957650,None,No SIMBAD match
4,138872235599735296,0.954226,None,No SIMBAD match
5,138940267880957184,0.952603,GALEX J030504.2+361533,WD*
6,138857843163156736,0.939332,None,No SIMBAD match
7,138995831873415680,0.927343,None,No SIMBAD match
8,138862378649972096,0.926817,Slee ACO 407 6,Rad
9,137341681053950592,0.921479,None,No SIMBAD match
